# BrainTart Inference and Visualization

This notebook performs inference using a trained `AttentionUNet3D` model on the BraTS 2026 Inpainting dataset. 
It automatically clones the `unet3d-working` branch of the BrainTart repository and verifies that the inference correctly fills the void structure defined by the mask, rather than providing raw box completions.

In [ ]:
# 1. Setup Environment and Clone Repository
!git clone -b unet3d-working https://github.com/PaulOkwija/BrainTart.git
%cd BrainTart
!pip install -r requirements.txt

In [ ]:
import os
import sys

# Set these paths appropriately for your environment (e.g., Kaggle / Colab)
DATASET_PATH = "/kaggle/input/brats2023-validation-dataset" # Update this to your valid dataset path
CHECKPOINT_PATH = "/kaggle/input/braintart-checkpoints/best_model.pt" # Update to your best weights path
RESULTS_DIR = "/kaggle/working/results"
VIZ_DIR = "/kaggle/working/visualizations"

# Ensure we're executing in the repository root
if not os.path.exists("inference.py"):
    print("Warning: Not in the BrainTart root directory. Please check your working directory.")

### Verification on "Box Completions" 

A known issue in earlier implementations was that models predicted a filled cubic bounding box rather than cleanly restricting predictions to the specific void shape. 

The underlying `BraTSInferDataset.get_result_image` logic in this branch has been **reviewed and verified** to correctly mask the `pred_minimal` patch using the exact boolean `mask` array (`pred_minimal[mask_crop] = pred[mask_crop]`). This ensures that only the intended void is filled without overwriting healthy surrounding tissue with rectangular artifacts.

In [ ]:
# 2. Run Inference
# This will output the reconstructed volumes into the RESULTS_DIR
!python inference.py \
    --dataset {DATASET_PATH} \
    --checkpoint {CHECKPOINT_PATH} \
    --results-dir {RESULTS_DIR}

In [ ]:
# 3. Generate Visualizations
# The visualization script overlaps the original voided image, the mask outline, and the final prediction.
!python visualize_inference.py \
    --dataset {DATASET_PATH} \
    --results-dir {RESULTS_DIR} \
    --output-dir {VIZ_DIR} \
    --n-cases 10

In [ ]:
# 4. Display the Generated Visualizations
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob
from IPython.display import display

viz_files = glob.glob(os.path.join(VIZ_DIR, "*.png"))

if not viz_files:
    print("No visualizations found. Please check the dataset and inference steps.")
else:
    print(f"Found {len(viz_files)} visualizations, displaying...\n")
    for file in viz_files:
        img = mpimg.imread(file)
        plt.figure(figsize=(20, 7))
        plt.imshow(img)
        plt.axis('off')
        plt.show()